[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/math/isospectral_drums/isospectral_drums.ipynb)

# Isospectral Drums

This example computes two drums of different shapes with exactly the same frequencies, the answer no to the question whether one can hear the shape of a drum. Each drum is cut from seven equal right triangles. A vibration of one drum, cut into its triangles, reflected and added three pieces at a time, is a vibration of the other, and it keeps its frequency.

The displacement of a drum is a field on its lattice of sites, and the transplantation is one map between the two drums' fields, `transplant = target_assembly(mixing(source_restriction))`: restriction maps read the source field's triangles, a signed mixing of the triangles makes the destination's pieces, and the adjoints of the destination's restriction maps put them in place. The transplant carries the wave operator across, `target_laplacian(transplant) == transplant(source_laplacian)`, so every standing wave of one drum goes to a standing wave of the other at the same frequency.

In spectral notation this relationship reads $L_{\mathrm{target}}T = TL_{\mathrm{source}}$.

In [ ]:
# The repository root on the path, for numga and the examples; in Colab, fetch the repository first.
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    root = Path("/content/numga")
    if not root.exists():
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/EelcoHoogendoorn/numga.git", str(root)], check=True)
else:
    root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "numga").is_dir() and (p / "examples").is_dir())
sys.path.insert(0, str(root))

In [ ]:
%matplotlib inline
from collections.abc import Iterator
import numpy as np

from numga import NumpyContext, concatenate, stack
from numga.algebras import VGA3D as ga
from numga.sparse import SparseExtensor
from examples.animation import save_figure, show_animation
from examples.math.isospectral_drums import render
from examples.math.isospectral_drums.numerics import drum, triangle

In [ ]:
context = NumpyContext(ga)
mv = context.multivector
Scalar = ga.gatype.scalar()
Vector = ga.gatype.vector()

## 1. Two drums from seven triangles

Each drum consists of seven congruent right triangles, with its outer edge held fixed. The vertex order aligns each triangle with the same reference triangle; moving a field between these local coordinates performs the required reflections and turns.

In [ ]:
n_lattice = 4
source_indices = np.array([
    [0, 1, 4], [5, 1, 4], [5, 9, 4], [5, 9, 6],
    [10, 9, 6], [10, 9, 14], [10, 11, 6],
])
target_indices = np.array([
    [5, 1, 4], [5, 9, 4], [8, 9, 4], [5, 9, 6],
    [10, 9, 6], [10, 11, 6], [10, 11, 14],
])

offsets = np.arange(n_lattice)
lattice = (mv.x * offsets[None, :] + mv.y * offsets[:, None]).reshape((-1,))  # [lattice vertices] Vector
source_tiles = lattice[source_indices]                              # [tiles, 3] Vector
target_tiles = lattice[target_indices]                              # [tiles, 3] Vector
tiles = stack((source_tiles, target_tiles))                           # [2, tiles, 3] Vector
save_figure(render.draw_tiles(tiles), "isospectral_drums_tiles");

## 2. The wave operator

The displacement at each interior sample couples to its four lattice neighbours. Boundary samples stay at zero. This lattice is preserved by the triangle reflections, so the discrete construction has the same exact transplantation identity as the continuous drums.

In [ ]:
# The samples along each tile's legs, the standing waves solved for, the three mixed into the vibration and
# their amplitudes, and the displacement at the ends of the colour scale.
n_subdivisions = 24
n_modes = 9
mixed_modes = np.array([0, 2, 5])
mix_amplitudes = np.array([0.30, 0.22, 0.26])
exposure = 0.45
spacing = 1 / n_subdivisions

In [ ]:
source_vertices, source_faces, source_neighbors, source_restriction, _, source_lift = drum(
    source_indices, n_lattice, n_subdivisions)
target_vertices, target_faces, target_neighbors, _, target_assembly, target_lift = drum(
    target_indices, n_lattice, n_subdivisions)

# Each neighbour contributes a displacement difference; missing neighbours have zero displacement.
source_laplacian = (4 * Scalar - source_neighbors) / spacing**2      # Scalar[source sites] <- Scalar[source sites]
target_laplacian = (4 * Scalar - target_neighbors) / spacing**2      # Scalar[target sites] <- Scalar[target sites]

# Solve for standing waves on the source drum.
# Inserting a zero boundary preserves the interior field's inner product.
measure = source_lift.adjoint()(source_lift)                        # Scalar[source sites] <- Scalar[source sites]
frequencies_squared, modes = source_laplacian.eigh(measure, n_modes) # [modes] Scalar, [modes] Scalar[source sites]
# Scaled by the spacing, the standing waves have displacements of order one.
waves = modes / spacing                                              # [modes] Scalar[source sites]

## 3. Transplantation

Each destination triangle receives three signed source pieces. The signs make their values agree at internal joins and cancel at the outer boundary. The adjoint of a triangle's read map inserts a field back into that triangle; summing these insertions builds the transplantation map.

In [ ]:
contributors = np.array([
    [1, 0, 6], [0, 2, 4], [1, 2, 3], [3, 0, 5],
    [4, 1, 5], [5, 2, 6], [4, 3, 6],
])
signs = np.array([
    [1, -1, 1], [1, 1, 1], [1, -1, 1], [1, -1, 1],
    [1, -1, -1], [1, -1, 1], [1, -1, -1],
])
target_tile = 0

In [ ]:
n_tiles = len(source_indices)
n_samples = source_restriction.shape[0] // n_tiles
tile_indices = np.arange(n_tiles * n_samples).reshape(n_tiles, n_samples)
read_tile = SparseExtensor.selection(context, tile_indices, n_tiles * n_samples) * Scalar  # [tiles] Scalar[samples] <- Scalar[tile samples]

# Select the source pieces, change their signs, and insert into the destination triangles.
signed_tiles = read_tile[contributors] * signs[..., None]           # [tiles, terms] Scalar[samples] <- Scalar[tile samples]
tile_mix = read_tile[:, None].adjoint()(signed_tiles).sum(axis=1).sum(axis=0)  # Scalar[tile samples] <- Scalar[tile samples]
transplant = target_assembly(tile_mix(source_restriction))           # Scalar[target sites] <- Scalar[source sites]

# Three signed contributions to one destination triangle, followed by their sum.
source_pieces = read_tile(source_restriction(waves[mixed_modes[-1]] * mix_amplitudes[-1]))   # [tiles] Scalar[samples]
contributions = source_pieces[contributors[target_tile]] * signs[target_tile]  # [terms] Scalar[samples]
pieces = concatenate((contributions, contributions.sum(axis=0)[None]), axis=0)  # [terms + 1] Scalar[samples]
local_vertices, local_faces = triangle(n_subdivisions)
save_figure(render.draw_pieces(local_vertices, local_faces, pieces, exposure), "isospectral_drums_pieces");

The first three panels are the signed contributions of a source wave to one destination triangle, and the fourth their sum: orange and blue are opposite displacements, and the thin lines where it is zero. The signs make the pieces agree along the triangle's inner edges and cancel along the drum's outer edge.

## 4. A vibration carried across

Three standing waves of the source drum, mixed, vibrate each at its own frequency, so the drum's motion never settles into one shape. At every instant its transplant is a vibration of the other drum, and since the transplant carries the wave operator across, the other drum moves in step at the same frequencies. The outlines are the seven triangles each drum is cut into. Equal frequencies do not make the two drums sound alike under any strike, since a strike excites the modes with different strengths. Drawing the animation takes about ten seconds.

In [ ]:
# The frames, over one period of the slowest wave mixed in, and their duration.
n_frames = 96
duration_ms = 60

In [ ]:
# Restore the fixed boundary for drawing the displacement fields.
def vibration() -> Iterator[tuple[Scalar, Scalar]]:
    """A mix of standing waves on the source drum, each at its own frequency, and its transplant on the
    target drum, frame by frame."""
    frequencies = frequencies_squared[mixed_modes].square_root().to_array()   # [mixed]
    times = np.linspace(0, 2 * np.pi / frequencies[0], n_frames, endpoint=False)
    weights = mix_amplitudes * np.cos(times[:, None] * frequencies)          # [frames, mixed]
    for frame_weights in weights:
        source = (waves[mixed_modes] * frame_weights).sum(axis=0)           # Scalar[source sites]
        yield source_lift(source), target_lift(transplant(source))


show_animation(render.frames((source_vertices, target_vertices), (source_faces, target_faces), tiles, vibration(),
                             exposure), "isospectral_drums", duration_ms)

In [ ]:
# checks
# The transplant carries the wave operator across: it commutes with it on every standing wave.
commutator = target_laplacian(transplant) - transplant(source_laplacian)     # Scalar[target sites] <- Scalar[source sites]
np.testing.assert_allclose(commutator(waves).to_array(), 0, atol=1e-6)

The seven-triangle construction is described in [Buser, Conway, Doyle and Semmler's *Some Planar Isospectral Domains*](https://arxiv.org/abs/1005.1839). The matched lattice and signed-piece recipe follow [Cleve Moler's worked transplantation example](https://blogs.mathworks.com/cleve/2012/08/20/can-one-hear-the-shape-of-a-drum-part-3-transplantation/); [Berkolaiko's folding diagrams](https://people.tamu.edu/~gberkolaiko/other/IsoOrigami/folding_drums.html) show how the cancellations arise.